# O3C - ML Stacking Ensemble Comparison

This notebook evaluates whether machine-learning stacking improves annual zone-mean CMIP6 ensemble performance relative to the O3A skill-weighted baseline.

Scope:
- Historical annual zone-mean evaluation against ERA5, 1985-2014.
- Variables: `pr`, `tasmax`, `tasmin`.
- Baselines: equal-weight multi-model mean and O3A skill-weighted mean.
- ML models: Random Forest, XGBoost if installed, Gradient Boosting fallback, and Ridge Regression stacking meta-learner.
- Validation: Grouped cross-validation by year, so all zones for held-out years are kept out of training together.

This is O3C. It does not claim the PINN physical-consistency component yet.

In [ ]:
# Cell 1 - Imports and configuration
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'output').exists() and (PROJECT_ROOT.parent / 'output').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUT_ROOT = PROJECT_ROOT / 'output' / 'o3c_ml_stacking_comparison'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
LOG_DIR = OUT_ROOT / 'logs'
for d in [TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

O2_TS_DIR = PROJECT_ROOT / 'output' / 'cmip6_eval' / 'timeseries'
O2_TABLE_DIR = PROJECT_ROOT / 'output' / 'cmip6_eval' / 'tables'
O3A_TABLE_DIR = PROJECT_ROOT / 'output' / 'o3a_skill_weighted_ensemble' / 'tables'

VARIABLES = ['pr', 'tasmax', 'tasmin']
RANDOM_STATE = 42
N_SPLITS = 5

print(f'[INFO] Project root: {PROJECT_ROOT}')
print(f'[INFO] Output root: {OUT_ROOT}')

In [ ]:
# Cell 2 - Helper functions

def ensure_year_column(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    if 'year' not in df.columns:
        if 'time' not in df.columns:
            raise KeyError(f'Expected year or time column. Found: {list(df.columns)}')
        df['year'] = pd.to_datetime(df['time']).dt.year
    df['year'] = df['year'].astype(int)
    return df


def kge(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) < 3 or np.std(obs) == 0 or np.std(sim) == 0:
        return np.nan
    r = np.corrcoef(obs, sim)[0, 1]
    alpha = np.std(sim, ddof=1) / np.std(obs, ddof=1)
    beta = np.mean(sim) / np.mean(obs) if np.mean(obs) != 0 else np.nan
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)


def compute_metrics(obs, pred):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]
    if len(obs) == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'bias': np.nan, 'r': np.nan, 'r2': np.nan, 'kge': np.nan, 'n': 0}
    rmse = mean_squared_error(obs, pred) ** 0.5
    mae = mean_absolute_error(obs, pred)
    bias = float(np.mean(pred - obs))
    r = np.corrcoef(obs, pred)[0, 1] if len(obs) >= 3 and np.std(obs) > 0 and np.std(pred) > 0 else np.nan
    r2 = r2_score(obs, pred) if len(obs) >= 2 else np.nan
    return {'rmse': rmse, 'mae': mae, 'bias': bias, 'r': r, 'r2': r2, 'kge': kge(obs, pred), 'n': len(obs)}


def try_make_xgb():
    try:
        from xgboost import XGBRegressor
        model = XGBRegressor(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='reg:squarederror',
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        return 'xgboost', model
    except Exception as e:
        with open(LOG_DIR / 'xgboost_fallback.txt', 'w', encoding='utf-8') as f:
            f.write(f'XGBoost unavailable, using GradientBoostingRegressor fallback. Reason: {type(e).__name__}: {e}\n')
        model = GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=250, learning_rate=0.04, max_depth=3)
        return 'gradient_boosting_fallback', model

print('[OK] helpers loaded')

In [ ]:
# Cell 3 - Load historical zone-mean data and O3A model weights
era5 = ensure_year_column(pd.read_csv(O2_TS_DIR / 'era5_zone_annual.csv'))
cmip = ensure_year_column(pd.read_csv(O2_TS_DIR / 'cmip6_historical_zone_annual.csv'))
o3a_weights = pd.read_csv(O3A_TABLE_DIR / 'o3a_model_pools_and_weights.csv')

# Keep only O3A model pools for consistency with the completed projection ensemble.
model_pools = {
    var: o3a_weights[o3a_weights['variable'] == var]['model'].tolist()
    for var in VARIABLES
}
print(model_pools)
print('[OK] data loaded')

In [ ]:
# Cell 4 - Build ML design matrices by variable
# Rows are annual zone observations. Model columns are CMIP6 model annual values.
# Additional predictors include zone one-hot encoding and normalized year.

design_tables = {}
feature_tables = {}

for var in VARIABLES:
    obs = era5[era5['variable'] == var][['year', 'zone', 'value']].rename(columns={'value': 'era5'})
    sim = cmip[(cmip['variable'] == var) & (cmip['model'].isin(model_pools[var]))][['year', 'zone', 'model', 'value']]
    wide = sim.pivot_table(index=['year', 'zone'], columns='model', values='value').reset_index()
    df = obs.merge(wide, on=['year', 'zone'], how='inner')
    model_cols = [m for m in model_pools[var] if m in df.columns]
    # Drop rows with missing model predictors for a fair historical comparison.
    df = df.dropna(subset=['era5'] + model_cols).reset_index(drop=True)
    zone_dummies = pd.get_dummies(df['zone'], prefix='zone', dtype=float)
    year_scaled = ((df['year'] - df['year'].mean()) / df['year'].std()).rename('year_scaled')
    X = pd.concat([df[model_cols].astype(float), zone_dummies, year_scaled], axis=1)
    y = df['era5'].astype(float)
    design_tables[var] = {'df': df, 'X': X, 'y': y, 'model_cols': model_cols}
    feature_tables[var] = pd.DataFrame({'feature': X.columns})
    df.to_csv(TABLE_DIR / f'o3c_design_table_{var}.csv', index=False)
    X.to_csv(TABLE_DIR / f'o3c_feature_matrix_{var}.csv', index=False)
    print(f'[OK] {var}: rows={len(df)}, model_cols={model_cols}, features={X.shape[1]}')

In [ ]:
# Cell 5 - Cross-validated baseline and ML predictions
all_predictions = []
all_metrics = []
feature_importance_rows = []

for var in VARIABLES:
    pack = design_tables[var]
    df = pack['df'].copy()
    X = pack['X']
    y = pack['y'].to_numpy(dtype=float)
    model_cols = pack['model_cols']
    groups = df['year'].to_numpy()
    unique_years = np.unique(groups)
    n_splits = min(N_SPLITS, len(unique_years))
    splitter = GroupKFold(n_splits=n_splits)
    weights = o3a_weights[o3a_weights['variable'] == var].set_index('model')['weight']
    weights = weights.loc[model_cols]
    weights = weights / weights.sum()

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        test_meta = df.iloc[test_idx][['year', 'zone']].copy()

        # Baselines
        equal_pred = X_test[model_cols].mean(axis=1).to_numpy(dtype=float)
        skill_pred = (X_test[model_cols] * weights.values).sum(axis=1).to_numpy(dtype=float)

        rf = RandomForestRegressor(n_estimators=400, max_depth=None, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
        xgb_name, xgb = try_make_xgb()
        ridge_direct = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 25)))

        rf.fit(X_train, y_train)
        xgb.fit(X_train, y_train)
        ridge_direct.fit(X_train, y_train)

        rf_pred = rf.predict(X_test)
        xgb_pred = xgb.predict(X_test)
        ridge_pred = ridge_direct.predict(X_test)

        # Stacking meta-learner: train on out-of-fold predictions within training split.
        inner = KFold(n_splits=min(4, len(train_idx)), shuffle=True, random_state=RANDOM_STATE)
        train_meta = np.zeros((len(train_idx), 3), dtype=float)
        for inner_train_rel, inner_val_rel in inner.split(X_train):
            Xi_tr, Xi_val = X_train.iloc[inner_train_rel], X_train.iloc[inner_val_rel]
            yi_tr = y_train[inner_train_rel]
            rf_i = RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
            xgb_name_i, xgb_i = try_make_xgb()
            ridge_i = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 25)))
            rf_i.fit(Xi_tr, yi_tr)
            xgb_i.fit(Xi_tr, yi_tr)
            ridge_i.fit(Xi_tr, yi_tr)
            train_meta[inner_val_rel, 0] = rf_i.predict(Xi_val)
            train_meta[inner_val_rel, 1] = xgb_i.predict(Xi_val)
            train_meta[inner_val_rel, 2] = ridge_i.predict(Xi_val)

        meta = RidgeCV(alphas=np.logspace(-4, 4, 25))
        meta.fit(train_meta, y_train)
        test_meta_features = np.column_stack([rf_pred, xgb_pred, ridge_pred])
        stack_pred = meta.predict(test_meta_features)

        methods = {
            'equal_weight_mean': equal_pred,
            'o3a_skill_weighted_mean': skill_pred,
            'random_forest': rf_pred,
            xgb_name: xgb_pred,
            'ridge_direct': ridge_pred,
            'ridge_stacking_meta': stack_pred,
        }

        for method, pred in methods.items():
            pred_df = test_meta.copy()
            pred_df['variable'] = var
            pred_df['fold'] = fold
            pred_df['method'] = method
            pred_df['era5'] = y_test
            pred_df['prediction'] = pred
            all_predictions.append(pred_df)
            metrics = compute_metrics(y_test, pred)
            metrics.update({'variable': var, 'fold': fold, 'method': method})
            all_metrics.append(metrics)

        # Feature importance from RF, fold-level
        for feat, imp in zip(X.columns, rf.feature_importances_):
            feature_importance_rows.append({'variable': var, 'fold': fold, 'model': 'random_forest', 'feature': feat, 'importance': imp})

predictions = pd.concat(all_predictions, ignore_index=True)
metrics_by_fold = pd.DataFrame(all_metrics)
feature_importance = pd.DataFrame(feature_importance_rows)

predictions.to_csv(TABLE_DIR / 'o3c_cross_validated_predictions.csv', index=False)
metrics_by_fold.to_csv(TABLE_DIR / 'o3c_metrics_by_fold.csv', index=False)
feature_importance.to_csv(TABLE_DIR / 'o3c_random_forest_feature_importance_by_fold.csv', index=False)

print(metrics_by_fold.head())
print(f'[OK] saved predictions and fold metrics')

In [ ]:
# Cell 6 - Summarize model comparison metrics
summary_rows = []
for (var, method), sub in predictions.groupby(['variable', 'method']):
    metrics = compute_metrics(sub['era5'], sub['prediction'])
    metrics.update({'variable': var, 'method': method})
    summary_rows.append(metrics)

metric_summary = pd.DataFrame(summary_rows)
metric_summary = metric_summary[['variable', 'method', 'n', 'kge', 'r', 'r2', 'rmse', 'mae', 'bias']]
metric_summary = metric_summary.sort_values(['variable', 'rmse']).reset_index(drop=True)
metric_summary.to_csv(TABLE_DIR / 'o3c_model_comparison_metric_summary.csv', index=False)

# Rank methods by variable using normalized RMSE, MAE, |bias|, KGE, and r.
rank_rows = []
for var, sub in metric_summary.groupby('variable'):
    tmp = sub.copy()
    for col, high_good in [('kge', True), ('r', True), ('rmse', False), ('mae', False), ('bias', False)]:
        vals = tmp[col].abs() if col == 'bias' else tmp[col]
        vmin, vmax = vals.min(), vals.max()
        if np.isclose(vmin, vmax):
            score = np.ones(len(tmp))
        else:
            score = (vals - vmin) / (vmax - vmin)
            if not high_good:
                score = 1 - score
        tmp[f'{col}_score'] = score
    tmp['composite_score'] = tmp[['kge_score', 'r_score', 'rmse_score', 'mae_score', 'bias_score']].mean(axis=1)
    tmp = tmp.sort_values('composite_score', ascending=False)
    rank_rows.append(tmp)
method_ranking = pd.concat(rank_rows, ignore_index=True)
method_ranking.to_csv(TABLE_DIR / 'o3c_method_ranking_by_variable.csv', index=False)
print(method_ranking[['variable','method','composite_score','kge','r','rmse','mae','bias']])
print('[OK] saved metric summary and method ranking')

In [ ]:
# Cell 7 - Final fit on all historical data and save interpretable artifacts
# These fitted models are for interpretation and potential later application. Cross-validated metrics remain the performance evidence.
final_rows = []
meta_rows = []

for var in VARIABLES:
    pack = design_tables[var]
    X = pack['X']
    y = pack['y'].to_numpy(dtype=float)
    rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
    xgb_name, xgb = try_make_xgb()
    ridge_direct = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 25)))
    rf.fit(X, y)
    xgb.fit(X, y)
    ridge_direct.fit(X, y)

    for feat, imp in zip(X.columns, rf.feature_importances_):
        final_rows.append({'variable': var, 'model': 'random_forest_final', 'feature': feat, 'importance': imp})

    base_pred = np.column_stack([rf.predict(X), xgb.predict(X), ridge_direct.predict(X)])
    meta = RidgeCV(alphas=np.logspace(-4, 4, 25))
    meta.fit(base_pred, y)
    for name, coef in zip(['random_forest', xgb_name, 'ridge_direct'], meta.coef_):
        meta_rows.append({'variable': var, 'meta_feature': name, 'ridge_meta_coefficient': coef, 'ridge_meta_intercept': meta.intercept_})

final_importance = pd.DataFrame(final_rows)
meta_coefficients = pd.DataFrame(meta_rows)
final_importance.to_csv(TABLE_DIR / 'o3c_final_random_forest_feature_importance.csv', index=False)
meta_coefficients.to_csv(TABLE_DIR / 'o3c_ridge_stacking_meta_coefficients.csv', index=False)
print(final_importance.head())
print(meta_coefficients)
print('[OK] saved final interpretability artifacts')

In [ ]:
# Cell 8 - Figures
metric_summary = pd.read_csv(TABLE_DIR / 'o3c_model_comparison_metric_summary.csv')
method_ranking = pd.read_csv(TABLE_DIR / 'o3c_method_ranking_by_variable.csv')
feature_importance = pd.read_csv(TABLE_DIR / 'o3c_final_random_forest_feature_importance.csv')

# Figure 1: RMSE comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, var in zip(axes, VARIABLES):
    sub = metric_summary[metric_summary['variable'] == var].sort_values('rmse', ascending=True)
    ax.barh(sub['method'], sub['rmse'], color='#4c78a8', edgecolor='black', linewidth=0.5)
    ax.invert_yaxis()
    ax.set_title(var.upper(), fontweight='bold')
    ax.set_xlabel('Cross-validated RMSE', fontweight='bold')
    ax.grid(axis='x', linestyle=':', alpha=0.5)
fig.suptitle('O3C Cross-Validated Ensemble Method Comparison', fontsize=14, fontweight='bold')
fig.savefig(FIG_DIR / 'o3c_method_comparison_rmse.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'o3c_method_comparison_rmse.pdf', bbox_inches='tight')
plt.show()

# Figure 2: composite ranking
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, var in zip(axes, VARIABLES):
    sub = method_ranking[method_ranking['variable'] == var].sort_values('composite_score', ascending=True)
    ax.barh(sub['method'], sub['composite_score'], color='#59a14f', edgecolor='black', linewidth=0.5)
    ax.set_title(var.upper(), fontweight='bold')
    ax.set_xlabel('Composite comparison score', fontweight='bold')
    ax.grid(axis='x', linestyle=':', alpha=0.5)
fig.suptitle('O3C Method Ranking by Variable', fontsize=14, fontweight='bold')
fig.savefig(FIG_DIR / 'o3c_method_composite_ranking.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'o3c_method_composite_ranking.pdf', bbox_inches='tight')
plt.show()

# Figure 3: top RF feature importance
for var in VARIABLES:
    sub = feature_importance[feature_importance['variable'] == var].sort_values('importance', ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(8, 4.8), constrained_layout=True)
    ax.barh(sub['feature'][::-1], sub['importance'][::-1], color='#f28e2b', edgecolor='black', linewidth=0.5)
    ax.set_title(f'O3C Random Forest Feature Importance: {var.upper()}', fontweight='bold')
    ax.set_xlabel('Importance', fontweight='bold')
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    fig.savefig(FIG_DIR / f'o3c_rf_feature_importance_{var}.png', dpi=350, bbox_inches='tight')
    fig.savefig(FIG_DIR / f'o3c_rf_feature_importance_{var}.pdf', bbox_inches='tight')
    plt.show()

print('[OK] figures saved')

In [ ]:
# Cell 9 - Completion summary
required = [
    TABLE_DIR / 'o3c_cross_validated_predictions.csv',
    TABLE_DIR / 'o3c_metrics_by_fold.csv',
    TABLE_DIR / 'o3c_model_comparison_metric_summary.csv',
    TABLE_DIR / 'o3c_method_ranking_by_variable.csv',
    TABLE_DIR / 'o3c_final_random_forest_feature_importance.csv',
    TABLE_DIR / 'o3c_ridge_stacking_meta_coefficients.csv',
    FIG_DIR / 'o3c_method_comparison_rmse.png',
    FIG_DIR / 'o3c_method_composite_ranking.png',
]
check = pd.DataFrame([{'path': str(p), 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0} for p in required])
check.to_csv(TABLE_DIR / 'o3c_completion_checklist.csv', index=False)

best = pd.read_csv(TABLE_DIR / 'o3c_method_ranking_by_variable.csv').sort_values(['variable', 'composite_score'], ascending=[True, False])
summary_lines = [
    '# O3C ML Stacking Ensemble Comparison Summary',
    '',
    '## Purpose',
    '',
    'This workflow compares ML stacking against equal-weight and O3A skill-weighted annual zone-mean ensemble baselines using ERA5 as the 1985-2014 reference.',
    '',
    '## Validation',
    '',
    'Grouped 5-fold cross-validation by year was used so that all hydroclimatic zones from held-out years were excluded from training.',
    '',
    '## Main Outputs',
    '',
    '- Cross-validated predictions: `tables/o3c_cross_validated_predictions.csv`',
    '- Fold metrics: `tables/o3c_metrics_by_fold.csv`',
    '- Metric summary: `tables/o3c_model_comparison_metric_summary.csv`',
    '- Method ranking: `tables/o3c_method_ranking_by_variable.csv`',
    '- Feature importance: `tables/o3c_final_random_forest_feature_importance.csv`',
    '- Ridge meta coefficients: `tables/o3c_ridge_stacking_meta_coefficients.csv`',
    '- Figures: `figures/`',
    '',
    '## Best Methods by Variable',
    '',
    best.groupby('variable').head(1).to_markdown(index=False),
    '',
    '## Scope Note',
    '',
    'This is an ML-stacking comparison for annual zone-mean historical reconstruction. It supports the ML component of O3 but does not yet implement the PINN physical-consistency component.',
    '',
]
(OUT_ROOT / 'O3C_ML_STACKING_COMPARISON_SUMMARY.md').write_text('\n'.join(summary_lines), encoding='utf-8')
print(check)
print(f'[OK] saved {OUT_ROOT / "O3C_ML_STACKING_COMPARISON_SUMMARY.md"}')

In [ ]:
# Cell 10 - Combined publication figures
# Clean figure titles without internal section labels.
metric_summary = pd.read_csv(TABLE_DIR / 'o3c_model_comparison_metric_summary.csv')
method_ranking = pd.read_csv(TABLE_DIR / 'o3c_method_ranking_by_variable.csv')
feature_importance = pd.read_csv(TABLE_DIR / 'o3c_final_random_forest_feature_importance.csv')

var_labels = {'pr': 'Precipitation', 'tasmax': 'Tmax', 'tasmin': 'Tmin'}
method_labels = {
    'equal_weight_mean': 'Equal weight',
    'o3a_skill_weighted_mean': 'Skill weighted',
    'random_forest': 'Random forest',
    'gradient_boosting_fallback': 'Gradient boosting',
    'xgboost': 'XGBoost',
    'ridge_direct': 'Ridge',
    'ridge_stacking_meta': 'Stacked ridge',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), constrained_layout=True)
for j, var in enumerate(VARIABLES):
    sub = metric_summary[metric_summary['variable'] == var].copy()
    sub['label'] = sub['method'].map(method_labels).fillna(sub['method'])
    sub_rmse = sub.sort_values('rmse', ascending=True)
    ax = axes[0, j]
    ax.barh(sub_rmse['label'], sub_rmse['rmse'], color='#4c78a8', edgecolor='black', linewidth=0.5)
    ax.invert_yaxis()
    ax.set_title(f"{chr(65+j)}. {var_labels[var]}", loc='left', fontweight='bold', fontsize=12)
    ax.set_xlabel('RMSE', fontweight='bold')
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    sub_kge = sub.sort_values('kge', ascending=True)
    ax = axes[1, j]
    ax.barh(sub_kge['label'], sub_kge['kge'], color='#59a14f', edgecolor='black', linewidth=0.5)
    ax.set_title(f"{chr(68+j)}. {var_labels[var]}", loc='left', fontweight='bold', fontsize=12)
    ax.set_xlabel('KGE', fontweight='bold')
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    for a in [axes[0, j], axes[1, j]]:
        for tick in a.get_xticklabels() + a.get_yticklabels():
            tick.set_fontweight('bold')
fig.savefig(FIG_DIR / 'ensemble_method_comparison_combined.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'ensemble_method_comparison_combined.pdf', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.7), constrained_layout=True)
for j, var in enumerate(VARIABLES):
    sub = method_ranking[method_ranking['variable'] == var].copy()
    sub['label'] = sub['method'].map(method_labels).fillna(sub['method'])
    sub = sub.sort_values('composite_score', ascending=True)
    axes[j].barh(sub['label'], sub['composite_score'], color='#8f63a8', edgecolor='black', linewidth=0.5)
    axes[j].set_title(f"{chr(65+j)}. {var_labels[var]}", loc='left', fontweight='bold', fontsize=12)
    axes[j].set_xlabel('Composite score', fontweight='bold')
    axes[j].grid(axis='x', linestyle=':', alpha=0.5)
    for tick in axes[j].get_xticklabels() + axes[j].get_yticklabels():
        tick.set_fontweight('bold')
fig.savefig(FIG_DIR / 'ensemble_method_composite_ranking_combined.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'ensemble_method_composite_ranking_combined.pdf', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), constrained_layout=True)
for j, var in enumerate(VARIABLES):
    sub = feature_importance[feature_importance['variable'] == var].copy().sort_values('importance', ascending=False).head(10)
    axes[j].barh(sub['feature'][::-1], sub['importance'][::-1], color='#f28e2b', edgecolor='black', linewidth=0.5)
    axes[j].set_title(f"{chr(65+j)}. {var_labels[var]}", loc='left', fontweight='bold', fontsize=12)
    axes[j].set_xlabel('Importance', fontweight='bold')
    axes[j].grid(axis='x', linestyle=':', alpha=0.5)
    axes[j].tick_params(axis='both', labelsize=8.5)
    for tick in axes[j].get_xticklabels() + axes[j].get_yticklabels():
        tick.set_fontweight('bold')
fig.savefig(FIG_DIR / 'random_forest_feature_importance_combined.png', dpi=350, bbox_inches='tight')
fig.savefig(FIG_DIR / 'random_forest_feature_importance_combined.pdf', bbox_inches='tight')
plt.show()

print('[OK] combined publication figures saved')